In [11]:
from PIL import Image, ImageDraw, ImageFont
import random

In [12]:
def _load_font(font_path, size):
    try:
        return ImageFont.truetype(font_path, size=size)
    except OSError:
        try:
            from matplotlib import font_manager
            p = font_manager.findfont("DejaVu Sans")
            print(f"[warn] '{font_path}' не найден, использую {p}")
            return ImageFont.truetype(p, size=size)
        except Exception:
            print("[warn] fallback на load_default")
            return ImageFont.load_default()


def fill_form_blocks(
    background_path,
    mask_path,
    font_path,
    output_path,
    x0, y0,
    w, h, dx, dy,
    cols, rows,
    block_dx, block_dy,
    n_blocks_x=1, n_blocks_y=1,
    threshold=128,
    invert=False,
    alphabet="АБВГДЕЁЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯ",
    font_size=None,
    jitter=1,
    text_color=(0, 0, 0, 255)
):
    img = Image.open(background_path).convert("RGBA")
    draw = ImageDraw.Draw(img)

    step_x = cols * w + (cols - 1) * dx + block_dx
    step_y = rows * h + (rows - 1) * dy + block_dy

    mask = Image.open(mask_path).convert("L")

    total_cols = n_blocks_x * cols
    total_rows = n_blocks_y * rows
    mask = mask.resize((total_cols, total_rows), resample=Image.BOX)
    if invert:
        mask = Image.eval(mask, lambda p: 255 - p)
    mask_bin = mask.point(lambda p: 255 if p < threshold else 0)
    mp = mask_bin.load()

    def is_black(bi, bj, ci, cj):
        return mp[bi * cols + ci, bj * rows + cj] != 0

    if font_size is None:
        font_size = int(h * 0.85)
    font = _load_font(font_path, font_size)

    total = 0
    for bj in range(n_blocks_y):
        for bi in range(n_blocks_x):
            for cj in range(rows):
                for ci in range(cols):
                    if not is_black(bi, bj, ci, cj):
                        continue
                    total += 1
                    cell_x = x0 + bi * step_x + ci * (w + dx)
                    cell_y = y0 + bj * step_y + cj * (h + dy)

                    char = random.choice(alphabet)
                    bbox = draw.textbbox((0, 0), char, font=font)
                    tw = bbox[2] - bbox[0]
                    th = bbox[3] - bbox[1]
                    pos_x = cell_x + (w - tw) / 2 - bbox[0] + random.randint(-jitter, jitter)
                    pos_y = cell_y + (h - th) / 2 - bbox[1] + random.randint(-jitter, jitter)

                    draw.text((pos_x, pos_y), char, font=font, fill=text_color)

    img.convert("RGB").save(output_path)

In [13]:
fill_form_blocks(
    background_path="blank_1.jpg",
    mask_path="mask.jpg",
    font_path="Caveat-VariableFont_wght.ttf",
    output_path="output.jpg",
    x0=88, y0=449,
    w=23, h=36,
    dx=8.2, dy=10,
    cols=17, rows=5,
    block_dx=58, block_dy=26,
    n_blocks_x=2, n_blocks_y=4,
    threshold=128,
    invert=False,
    jitter=0,
    font_size=28
)

In [14]:
from PIL import Image, ImageDraw, ImageFont
import random
from wonderwords import RandomWord


_rw = RandomWord()

# цифры для fallback
DIGITS = "0123456789"


def _load_font(font_path, size):
    try:
        return ImageFont.truetype(font_path, size=size)
    except OSError:
        try:
            from matplotlib import font_manager
            p = font_manager.findfont("DejaVu Sans")
            print(f"[warn] '{font_path}' не найден, использую {p}")
            return ImageFont.truetype(p, size=size)
        except Exception:
            print("[warn] fallback на load_default")
            return ImageFont.load_default()


def _random_word(length):
    if length <= 0:
        return None
    try:
        return _rw.word(word_min_length=length, word_max_length=length)
    except Exception:
        return None


def _digits(length):
    return "".join(random.choice(DIGITS) for _ in range(length))


def _word_or_fallback(L, uppercase=True, split_long_runs=True, max_word_len=12):
    if L <= 0:
        return ""
    if not split_long_runs or L <= max_word_len:
        w = _random_word(L)
        return (w.upper() if uppercase else w.lower()) if w else _digits(L)
    parts = []
    rem = L
    while rem > 0:
        chunk = min(rem, max_word_len)
        w = _random_word(chunk)
        parts.append((w.upper() if uppercase else w.lower()) if w else _digits(chunk))
        rem -= chunk
    return "".join(parts)


def _segments_in_block(is_black, bi, bj, cols, cj):
    segs = []
    ci = 0
    while ci < cols:
        if not is_black(bi, bj, ci, cj):
            ci += 1
            continue
        s = ci
        while ci < cols and is_black(bi, bj, ci, cj):
            ci += 1
        segs.append((s, ci))
    return segs


def fill_form_blocks(
    background_path,
    mask_path,
    font_path,
    output_path,
    x0, y0,
    w, h, dx, dy,
    cols, rows,
    block_dx, block_dy,
    n_blocks_x=1, n_blocks_y=1,
    threshold=128,
    invert=False,
    font_size=None,
    jitter=1,
    text_color=(0, 0, 0, 255),
    use_words=True,
    uppercase=True,
    split_long_runs=True,
    max_word_len=12,
):
    img = Image.open(background_path).convert("RGBA")
    draw = ImageDraw.Draw(img)

    step_x = cols * w + (cols - 1) * dx + block_dx
    step_y = rows * h + (rows - 1) * dy + block_dy

    mask = Image.open(mask_path).convert("L")
    total_cols = n_blocks_x * cols
    total_rows = n_blocks_y * rows
    mask = mask.resize((total_cols, total_rows), resample=Image.BOX)
    if invert:
        mask = Image.eval(mask, lambda p: 255 - p)
    mask_bin = mask.point(lambda p: 255 if p < threshold else 0)
    mp = mask_bin.load()

    def is_black(bi, bj, ci, cj):
        return mp[bi * cols + ci, bj * rows + cj] != 0

    if font_size is None:
        font_size = int(h * 0.85)
    font = _load_font(font_path, font_size)

    total = 0
    for bj in range(n_blocks_y):
        for bi in range(n_blocks_x):
            for cj in range(rows):
                for (s, e) in _segments_in_block(is_black, bi, bj, cols, cj):
                    L = e - s
                    chars = _word_or_fallback(
                        L, uppercase=uppercase,
                        split_long_runs=split_long_runs,
                        max_word_len=max_word_len,
                    ) if use_words else _digits(L)

                    for i, ch in enumerate(chars):
                        ci = s + i
                        total += 1
                        cell_x = x0 + bi * step_x + ci * (w + dx)
                        cell_y = y0 + bj * step_y + cj * (h + dy)

                        bbox = draw.textbbox((0, 0), ch, font=font)
                        tw = bbox[2] - bbox[0]
                        th = bbox[3] - bbox[1]
                        pos_x = cell_x + (w - tw) / 2 - bbox[0] + random.randint(-jitter, jitter)
                        pos_y = cell_y + (h - th) / 2 - bbox[1] + random.randint(-jitter, jitter)

                        draw.text((pos_x, pos_y), ch, font=font, fill=text_color)

    img.convert("RGB").save(output_path)
    print(f"[done] {output_path}  (клеток: {total})")

In [ ]:
fill_form_blocks(
    background_path="blank_1.jpg",
    mask_path="mask.jpg",
    font_path="Caveat-VariableFont_wght.ttf",
    output_path="output.jpg",
    x0=88, y0=449,
    w=23, h=36,
    dx=8.2, dy=10,
    cols=17, rows=5,
    block_dx=58, block_dy=26,
    n_blocks_x=2, n_blocks_y=4,
    threshold=128,
    invert=False,
    jitter=0,
    font_size=28,
    use_words=True
)

[done] output.jpg  (клеток: 209)
